In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
path = r"C:\Diploma\Datasets"

instacart = pd.read_csv(path + r"\instacart_full.csv")
superstore = pd.read_csv(path + r"\Global_Superstore2.csv", encoding="latin1")
amazon = pd.read_csv(path + r"\amazon.csv", encoding="utf-8", on_bad_lines="skip")
electronics = pd.read_excel(path + r"\electronika_03_04.xlsx")

In [3]:
datasets = {
    "Instacart": instacart,
    "Superstore": superstore,
    "Amazon": amazon,
    "Electronics": electronics
}

В рамках обзора для каждого датасета выполняется:
- описание структуры: размер, перечень и типы признаков;
- оценка качества данных: пропуски, дубликаты, базовая статистика, выбросы;
- первичная интерпретация с точки зрения задач:
  - сегментация клиентов;
  - модель оттока (churn);
  - персонализация предложений и потенциальный uplift-анализ.

## Общие вспомогательные функции

Далее определены функции для единообразного описания каждого датасета:

- `basic_description` — базовое структурное описание (размер, типы данных, список колонок, уникальность признаков, первые строки);
- `data_quality` — оценка качества данных (пропуски, пустые строки, дубликаты, базовая статистика и выбросы).

Один и тот же набор функций применяется ко всем датасетам, что позволяет сравнивать их между собой по одинаковым критериям.


In [4]:
def basic_description(df, name):
    print(f"\n{'='*40}\n БАЗОВОЕ ОПИСАНИЕ ДАТАСЕТА: {name}\n{'='*40}")
    
    print("\n▶ Форма датасета:")
    print(df.shape)

    print("\n▶ Типы данных:")
    display(df.dtypes)

    print("\n▶ Список колонок:")
    for col in df.columns:
        print(" -", col)

    print("\n▶ Уникальность колонок (top-20):")
    uniq = pd.DataFrame({
        "unique_count": df.nunique(),
        "unique_%": (df.nunique() / len(df) * 100).round(2),
        "dtype": df.dtypes
    }).sort_values("unique_%", ascending=False)

    display(uniq.head(20))

    print("\n▶ Первые 5 строк:")
    display(df.head())


In [5]:
def data_quality(df, name):
    print(f"\n{'='*40}\n КАЧЕСТВО ДАННЫХ: {name}\n{'='*40}")

    print("\n▶ Пропуски:")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if len(missing) == 0:
        print("Нет пропусков")
    else:
        display(pd.DataFrame({
            "missing_count": missing,
            "missing_%": (missing/len(df)*100).round(2)
        }))

    print("\n Пустые строки:")
    empty = (df == "").sum()
    empty = empty[empty > 0].sort_values(ascending=False)
    if len(empty) == 0:
        print("Нет пустых строк")
    else:
        display(pd.DataFrame(empty, columns=["empty_string_count"]))

    print("\n Дубликаты:")
    dups = df.duplicated().sum()
    print(f"Количество дублей строк: {dups}")

    print("\n Числовая статистика:")
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(num_cols) > 0:
        display(df[num_cols].describe().T)
    else:
        print("Числовых колонок нет.")

    print("\n Выбросы (1% - 99%):")
    if len(num_cols) > 0:
        q = df[num_cols].quantile([0.01, 0.99]).T
        q.columns = ['q01', 'q99']
        display(q)


In [6]:
basic_description(instacart, "Instacart")
data_quality(instacart, "Instacart")


 БАЗОВОЕ ОПИСАНИЕ ДАТАСЕТА: Instacart

▶ Форма датасета:
(33819106, 15)

▶ Типы данных:


order_id                    int64
product_id                  int64
add_to_cart_order           int64
reordered                   int64
user_id                     int64
eval_set                   object
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
product_name               object
aisle_id                    int64
department_id               int64
aisle                      object
department                 object
dtype: object


▶ Список колонок:
 - order_id
 - product_id
 - add_to_cart_order
 - reordered
 - user_id
 - eval_set
 - order_number
 - order_dow
 - order_hour_of_day
 - days_since_prior_order
 - product_name
 - aisle_id
 - department_id
 - aisle
 - department

▶ Уникальность колонок (top-20):


,unique_count,unique_%,dtype
order_id,3346083,9.89,int64
user_id,206209,0.61,int64
product_id,49685,0.15,int64
product_name,49685,0.15,object
add_to_cart_order,145,0.00,int64
eval_set,2,0.00,object
reordered,2,0.00,int64
order_number,100,0.00,int64
order_dow,7,0.00,int64
order_hour_of_day,24,0.00,int64



▶ Первые 5 строк:


...

,missing_count,missing_%
days_since_prior_order,2078068,6.14



 Пустые строки:
Нет пустых строк

 Дубликаты:
Количество дублей строк: 0

 Числовая статистика:


,count,mean,std,min,25%,50%,75%,max
order_id,33819106.0,1.710566e+06,987400.761933,1.0,855413.0,1710660.0,2565587.0,3421083.0
product_id,33819106.0,2.557551e+04,14097.696774,1.0,13519.0,25256.0,37935.0,49688.0
add_to_cart_order,33819106.0,8.367738e+00,7.139540,1.0,3.0,6.0,11.0,145.0
reordered,33819106.0,5.900617e-01,0.491822,0.0,0.0,1.0,1.0,1.0
user_id,33819106.0,1.029444e+05,59467.333955,1.0,51435.0,102626.0,154412.0,206209.0
order_number,33819106.0,1.713998e+01,17.498287,1.0,5.0,11.0,24.0,100.0
order_dow,33819106.0,2.737285e+00,2.093296,0.0,1.0,3.0,5.0,6.0
order_hour_of_day,33819106.0,1.343123e+01,4.246149,0.0,10.0,13.0,16.0,23.0
days_since_prior_order,31741038.0,1.136415e+01,8.940500,0.0,5.0,8.0,15.0,30.0
aisle_id,33819106.0,7.121799e+01,38.198982,1.0,31.0,83.0,107.0,134.0



 Выбросы (1% - 99%):


,q01,q99
order_id,34188.0,3387069.0
product_id,469.0,49235.0
add_to_cart_order,1.0,33.0
reordered,0.0,1.0
user_id,2192.0,204103.0
order_number,1.0,81.0
order_dow,0.0,6.0
order_hour_of_day,1.0,23.0
days_since_prior_order,0.0,30.0
aisle_id,3.0,130.0


Комментарии к структуре и качеству данных Instacart

**Размер и структура.**  
Датасет Instacart содержит около 33,8 млн строк и 15 столбцов.
Ключевые признаки:
- `user_id` - более 200 тыс. уникальных пользователей;
- `order_id` - свыше 3,3 млн уникальных заказов;
- `product_id` и `product_name` - около 50 тыс. уникальных товаров;
- `order_number`,`order_dow`,`order_hour_of_day`,`days_since_prior_order` - задают временную динамику и порядок заказов;
- `aisle`,`department` - иерархия товарных категорий.

**Качество данных.**  
Дубликатов нет, пустых строк нет.  
Пропуски присутствуют только в поле `days_since_prior_order` (~6 % строк), что логично интерпретируется как первый заказ пользователя (интервал до предыдущего заказа отсутствует по смыслу).

**Числовая статистика и выбросы.**  
Распределения числовых признаков (номер заказа, день недели, час, интервал между заказами) находятся в ожидаемых диапазонах. 1-й и 99-й процентили не показывают аномально больших значений, что упрощает работу чистки выбросов.

**Вывод.**  
Instacart представляет собой крупный, хорошо структурированный и качественный  датасет с богатой поведенческой информацией о клиентах и полной историей их заказов.  
Этот набор данных даёт возможность:
- строить поведенческие признаки (частота, давность, регулярность, ширина корзины);
- анализировать реакции клиентов на категории/товары;
- моделировать жизненный цикл клиента и прогнозировать отток.

С точки зрения задач диплома Instacart является наиболее перспективным датасетом.

In [7]:
basic_description(superstore, "Superstore")
data_quality(superstore, "Superstore")


 БАЗОВОЕ ОПИСАНИЕ ДАТАСЕТА: Superstore

▶ Форма датасета:
(51290, 24)

▶ Типы данных:


Row ID              int64
Order ID           object
Order Date         object
Ship Date          object
Ship Mode          object
Customer ID        object
Customer Name      object
Segment            object
City               object
State              object
Country            object
Postal Code       float64
Market             object
Region             object
Product ID         object
Category           object
Sub-Category       object
Product Name       object
Sales             float64
Quantity            int64
Discount          float64
Profit            float64
Shipping Cost     float64
Order Priority     object
dtype: object


▶ Список колонок:
 - Row ID
 - Order ID
 - Order Date
 - Ship Date
 - Ship Mode
 - Customer ID
 - Customer Name
 - Segment
 - City
 - State
 - Country
 - Postal Code
 - Market
 - Region
 - Product ID
 - Category
 - Sub-Category
 - Product Name
 - Sales
 - Quantity
 - Discount
 - Profit
 - Shipping Cost
 - Order Priority

▶ Уникальность колонок (top-20):


,unique_count,unique_%,dtype
Row ID,51290,100.00,int64
Order ID,25035,48.81,object
Profit,24575,47.91,float64
Sales,22995,44.83,float64
Product ID,10292,20.07,object
Shipping Cost,10037,19.57,float64
Product Name,3788,7.39,object
City,3636,7.09,object
Customer ID,1590,3.10,object
Ship Date,1464,2.85,object



▶ Первые 5 строк:


...

,missing_count,missing_%
Postal Code,41296,80.51



 Пустые строки:
Нет пустых строк

 Дубликаты:
Количество дублей строк: 0

 Числовая статистика:


,count,mean,std,min,25%,50%,75%,max
Row ID,51290.0,25645.500000,14806.291990,1.000,12823.250000,25645.500,38467.7500,51290.000
Postal Code,9994.0,55190.379428,32063.693350,1040.000,23223.000000,56430.500,90008.0000,99301.000
Sales,51290.0,246.490581,487.565361,0.444,30.758625,85.053,251.0532,22638.480
Quantity,51290.0,3.476545,2.278766,1.000,2.000000,3.000,5.0000,14.000
Discount,51290.0,0.142908,0.212280,0.000,0.000000,0.000,0.2000,0.850
Profit,51290.0,28.610982,174.340972,-6599.978,0.000000,9.240,36.8100,8399.976
Shipping Cost,51290.0,26.375915,57.296804,0.000,2.610000,7.790,24.4500,933.570



 Выбросы (1% - 99%):


,q01,q99
Row ID,513.89000,50777.11000
Postal Code,2149.00000,98115.00000
Sales,3.69000,2301.00000
Quantity,1.00000,11.00000
Discount,0.00000,0.70000
Profit,-351.50565,587.35995
Shipping Cost,0.20000,286.75430


Комментарии к структуре и качеству данных Superstore

**Размер и структура.**  
Датасет содержит около 51 тыс. строк и 24 столбца. Каждая строка описывает товарную позицию в заказе.  
Ключевые поля:
- `Customer ID` — около 1,5 тыс. уникальных клиентов;
- `Order ID` — примерно 25 тыс. уникальных заказов;
- `Product ID` — более 10 тыс. товаров;
- `Category`, `Sub-Category` — товарные группы;
- `Sales`, `Profit`, `Discount`, `Shipping Cost` — денежные и операционные показатели;
- `Order Date`, `Ship Date` — даты оформления и отправки заказа.

**Качество данных.**  
Дубликатов нет  
Основная проблема качества - высокий уровень пропусков в колонке `Postal Code` (более 80 % отсутствующих значений). Остальные поля заполнены значительно лучше.

**Числовая статистика и выбросы.**  
Продажи, прибыль и скидки имеют ожидаемые диапазоны, но распределения сильно правосторонние (есть крупные заказы с высокими значениями `Sales` и `Profit`), что отражается на 99-м процентиле. Это нужно будет учитывать при построении моделей и визуализаций, но в целом ситуация нормальная для финансовых показателей.

**Вывод.**  
Superstore подходит для задач:
- базового бизнес-анализа;
- расчёта RFM-признаков;
- простой сегментации клиентов по ценности и активности.

Однако по сравнению с Instacart датасет заметно меньше по числу клиентов и глубине истории заказов, из-за чего его потенциал для полноценного ML-пайплайна (особенно сложных моделей оттока и персонализации) ограничен.


In [ ]:

basic_description(amazon, "Amazon")
data_quality(amazon, "Amazon")


 БАЗОВОЕ ОПИСАНИЕ ДАТАСЕТА: Amazon

▶ Форма датасета:
(1465, 16)

▶ Типы данных:


product_id             object
product_name           object
category               object
discounted_price       object
actual_price           object
discount_percentage    object
rating                 object
rating_count           object
about_product          object
user_id                object
user_name              object
review_id              object
review_title           object
review_content         object
img_link               object
product_link           object
dtype: object


▶ Список колонок:
 - product_id
 - product_name
 - category
 - discounted_price
 - actual_price
 - discount_percentage
 - rating
 - rating_count
 - about_product
 - user_id
 - user_name
 - review_id
 - review_title
 - review_content
 - img_link
 - product_link

▶ Уникальность колонок (top-20):


,unique_count,unique_%,dtype
product_link,1465,100.00,object
img_link,1412,96.38,object
product_id,1351,92.22,object
product_name,1337,91.26,object
about_product,1293,88.26,object
review_content,1212,82.73,object
user_name,1194,81.50,object
review_id,1194,81.50,object
user_id,1194,81.50,object
review_title,1194,81.50,object



▶ Первые 5 строк:


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...



 КАЧЕСТВО ДАННЫХ: Amazon

▶ Пропуски:


,missing_count,missing_%
rating_count,2,0.14



 Пустые строки:
Нет пустых строк

 Дубликаты:
Количество дублей строк: 0

 Числовая статистика:
Числовых колонок нет.

 Выбросы (1% - 99%):


Комментарии к структуре и качеству данных Amazon

**Размер и структура.**  
Датасет содержит около 1,5 тыс. строк и 16 столбцов. Основные поля:
- `product_id`, `product_name`, `category` — идентификация товара;
- `discounted_price`, `actual_price`, `discount_percentage` — ценовая информация;
- `rating`, `rating_count` — агрегированные оценки;
- `user_id`, `user_name` — информация о пользователях;
- `review_id`, `review_title`, `review_content` — текстовые отзывы;
- `img_link`, `product_link` — ссылки на изображение и карточку товара.

**Качество данных.**  
Пропусков почти нет (небольшое количество в `rating_count`), дубликатов нет.  
Однако практически все признаки имеют тип `object`, включая те, что логически являются числовыми (цены, рейтинг, количество оценок). Это требует конвертации в числовой формат перед любыми количественными расчётами.

**Ограничения для задач диплома.**  
Несмотря на относительно чистые данные, этот датасет:
- не является транзакционным (нет истории заказов, последовательности покупок);
- не содержит явной временной оси и последовательности действий пользователей;
- описывает товары и отзывы, а не поведение клиентов во времени.

В результате он не подходит для задач диплома. Его можно использовать только как вспомогательный контентный источник (например, для анализа текстовых отзывов), но не как основной датасет для диплома.


In [9]:
basic_description(electronics, "Electronics")
data_quality(electronics, "Electronics")


 БАЗОВОЕ ОПИСАНИЕ ДАТАСЕТА: Electronics

▶ Форма датасета:
(357036, 37)

▶ Типы данных:


Order_ID                object
Email_new               object
Phone_new               object
Source                  object
OrderDate       datetime64[ns]
время                   object
месяц                    int64
ChangeDate              object
DeliveryDate            object
PaymentDate             object
Status                  object
Status_ID                int64
OneClick                 int64
CancelReason            object
Actions                 object
DeliveryType            object
PaymentType             object
Region                  object
Area                    object
Store_ID                 int64
FullSum                float64
Discount               float64
IM_Rozn_Sum            float64
Row_ID                 float64
Articul                float64
Nom_Name                object
NomGroup                object
Quant                  float64
RowPrice               float64
RowDiscount            float64
RowSum                 float64
Brand                   object
TN      


▶ Список колонок:
 - Order_ID
 - Email_new
 - Phone_new
 - Source
 - OrderDate
 - время
 - месяц
 - ChangeDate
 - DeliveryDate
 - PaymentDate
 - Status
 - Status_ID
 - OneClick
 - CancelReason
 - Actions
 - DeliveryType
 - PaymentType
 - Region
 - Area
 - Store_ID
 - FullSum
 - Discount
 - IM_Rozn_Sum
 - Row_ID
 - Articul
 - Nom_Name
 - NomGroup
 - Quant
 - RowPrice
 - RowDiscount
 - RowSum
 - Brand
 - TN
 - TK
 - NomFullPath
 - Week
 - Nom_ID

▶ Уникальность колонок (top-20):


,unique_count,unique_%,dtype
Order_ID,166794,46.72,object
ChangeDate,155189,43.47,object
Phone_new,123135,34.49,object
PaymentDate,105118,29.44,object
Email_new,99284,27.81,object
время,58549,16.40,object
Articul,26930,7.54,float64
Nom_ID,26928,7.54,float64
Nom_Name,26888,7.53,object
IM_Rozn_Sum,20622,5.78,float64



▶ Первые 5 строк:


...

,missing_count,missing_%
Actions,261496,73.24
CancelReason,230868,64.66
Brand,166141,46.53
TN,166101,46.52
TK,166101,46.52
DeliveryType,8285,2.32
Area,1631,0.46
Nom_Name,58,0.02
RowPrice,20,0.01
FullSum,20,0.01



 Пустые строки:
Нет пустых строк

 Дубликаты:
Количество дублей строк: 10

 Числовая статистика:


,count,mean,std,min,25%,50%,75%,max
месяц,357036.0,2.016035e+05,0.499751,201603.0,201603.0,201603.0,201604.0,201604.0
Status_ID,357036.0,1.508246e+01,1.723985,1.0,14.0,14.0,17.0,26.0
OneClick,357036.0,8.676716e-02,0.281494,0.0,0.0,0.0,0.0,1.0
Store_ID,357036.0,3.574291e+03,1257.634055,0.0,3088.0,3458.0,3668.0,9999.0
FullSum,357016.0,1.193250e+04,17900.956758,0.0,2369.0,6090.0,14850.0,888740.0
Discount,357016.0,1.869053e+02,875.940145,0.0,0.0,0.0,0.0,40011.0
IM_Rozn_Sum,357016.0,1.212027e+04,18178.124191,0.0,2340.0,6099.0,14998.0,888740.0
Row_ID,357016.0,1.679418e+00,0.966818,1.0,1.0,2.0,2.0,41.0
Articul,357016.0,1.168068e+06,53102.751256,11510.0,1157790.0,1157790.0,1195240.0,1263264.0
Quant,357016.0,1.037225e+00,1.050080,0.0,1.0,1.0,1.0,285.0



 Выбросы (1% - 99%):


,q01,q99
месяц,201603.0,201604.0
Status_ID,14.0,17.0
OneClick,0.0,1.0
Store_ID,2189.0,9999.0
FullSum,255.0,79999.0
Discount,0.0,3680.8
IM_Rozn_Sum,0.0,81899.0
Row_ID,1.0,5.0
Articul,1085371.0,1247183.0
Quant,1.0,2.0


Комментарии к структуре и качеству данных Electronics

**Размер и структура.**  
Датасет содержит около 357 тыс. строк и 37 столбцов. Присутствуют поля:
- идентификаторы заказов (`Order_ID`), товаров (`Nom_ID`, `Articul`), строк (`Row_ID`);
- контактные данные (`Email_new`, `Phone_new`);
- временные метки (`OrderDate`, `ChangeDate`, `DeliveryDate`, `PaymentDate`, `месяц`, `Week`);
- статусы и типы (`Status`, `Status_ID`, `DeliveryType`, `PaymentType`);
- финансовые показатели (`FullSum`, `IM_Rozn_Sum`, `RowPrice`, `RowDiscount`, `RowSum`);
- товарные атрибуты (`Nom_Name`, `Brand`, `NomFullPath`, `TN`, `TK`).

**Качество данных.**  
Несмотря на большой объём, качество данных является серьёзной проблемой:
- в ряде важных признаков доля пропусков очень высока (например, `Actions` — более 70 %, `CancelReason`, `Brand`, `TN` — около 45–65 %);
- присутствует небольшое количество дубликатов (10 строк);
- некоторые числовые признаки имеют выраженные выбросы, что видно по 1-му и 99-му процентилям.

Такой уровень «шума» и разреженности сильно усложняет подготовку данных и повышает риск того, что часть моделей будет основана на очень неполной информации.

**Вывод.**  
Несмотря на кажущуюся близость к реальному бизнес-кейсу, данный датасет требует масштабной очистки и реконструкции логики пользовательской истории. В совокупности с большим количеством пропусков это делает его менее подходящим для задач диплома, чем Instacart, где структура поведения клиента задана значительно чище и полнее.


5. Сравнение датасетов и итоговый выбор

В результате проведённого анализа я сравнила четыре датасета по одинаковым критериям: размер, структура, наличие временной динамики, качество данных и пригодность для задач сегментации, churn и персонализации.

Кратко:
- **Instacart** — крупный, хорошо структурированный транзакционный датасет с длинной историей заказов, минимальными проблемами качества и богатой поведенческой информацией по клиентам.
- **Superstore** — компактный и удобный для бизнес-аналитики набор, но с ограниченным числом клиентов и меньшей глубиной истории.
- **Amazon** — фактически отзывный, а не транзакционный датасет, без последовательности действий клиентов.
- **Electronics** — объёмный, но с высоким уровнем пропусков и неоднородной структурой, что сильно усложняет применение ML-подходов.

С учётом целей дипломной работы (анализ поведения клиентов, сегментация, модель оттока, персонализация и возможный uplift) в качестве основного источника данных выбран датасет **Instacart**. Далее вся последующая разработка моделей и признаков будет вестись на его основе.
